In [2]:
# import libraries
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
# NEW!
import sys

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

In [20]:
from data import data_reader, data_loader
from utils import helpers

In [53]:
class CAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(7,64)
        ### encoder layer
        self.encoding = nn.Linear(64,4)
        ### bottleneck layer
        self.bottleneck = nn.Linear(4,64)
        ### decoder layer
        self.decoding = nn.Linear(64,7)

    def forward(self,x):
        x = F.relu(self.input(x))
        x = F.relu(self.encoding(x))
        x = F.relu(self.bottleneck(x))
        # y = torch.sigmoid( self.decoding(x) )
        y = self.decoding(x)
        return x, y


In [54]:
a = CAE()
print(a)

CAE(
  (input): Linear(in_features=7, out_features=64, bias=True)
  (encoding): Linear(in_features=64, out_features=4, bias=True)
  (bottleneck): Linear(in_features=4, out_features=64, bias=True)
  (decoding): Linear(in_features=64, out_features=7, bias=True)
)


In [55]:
config = helpers.Config()
cfg = config.from_json("data")
data_read = data_reader.DataReader()
X, y = data_read.load_standardize_data('wind_forecast_2009')
data_load = data_loader.DataModelLoader(X, y)
train_loader = data_load.all_data_loader()

In [56]:
# loss function
loss_function = nn.MSELoss()

In [57]:
# def loss_function(W, x, recons_x, h, lam):
#     mse = lossfun(recons_x, x)
#     # Since: W is shape of N_hidden x N. So, we do not need to transpose it as
#     # opposed to #1
#     dh = h * (1 - h) # Hadamard product produces size N_batch x N_hidden
#     # Sum through the input dimension to improve efficiency, as suggested in #1
#     w_sum = torch.sum(Variable(W)**2, dim=1)
#     # unsqueeze to avoid issues with torch.mv
#     w_sum = w_sum.unsqueeze(1) # shape N_hidden x 1
#     contractive_loss = torch.sum(torch.mm(dh**2, w_sum), 0)
#     return mse + contractive_loss.mul_(lam)

In [58]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: ", device)

device:  cuda


In [59]:
model = CAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)

In [60]:

def train(epoch):
    model.train()
    train_loss = 0

    for idx, (data1, label) in enumerate(train_loader):

        data = torch.cat((data1, label), dim=1)
        data = data.to(device)

        encoded, recons_x = model(data)

        W = model.state_dict()['input.weight']
        # loss = loss_function(recons_x,data)
        loss = loss_function(W,data,recons_x,encoded,1e-4)

        optimizer.zero_grad()
        # Get the weights
        # model.state_dict().keys()
        # change the key by seeing the keys manually.
        # (In future I will try to make it automatic)

        loss.backward()
        train_loss += loss.item()
        optimizer.step()

        # if idx % 5 == 0:
        #     print('Train epoch: {} [{}/{}({:.0f}%)]\t Loss: {:.6f}'.format(
        #           epoch, idx*len(data), len(train_loader),
        #           100*idx/len(train_loader),
        #           loss.data[0]/len(data)))


    print('====> Epoch: {} Average loss: {:.4f}'.format(
         epoch, train_loss / len(train_loader)))
    # model.samples_write(data,epoch)


In [68]:
for epoch in range(50):
    train(epoch)

====> Epoch: 0 Average loss: 5811.7819
====> Epoch: 1 Average loss: 4093.3813
====> Epoch: 2 Average loss: 3045.8539
====> Epoch: 3 Average loss: 2304.5195
====> Epoch: 4 Average loss: 1694.5592
====> Epoch: 5 Average loss: 1214.9056
====> Epoch: 6 Average loss: 862.6867
====> Epoch: 7 Average loss: 618.8479
====> Epoch: 8 Average loss: 446.9239
====> Epoch: 9 Average loss: 331.0076
====> Epoch: 10 Average loss: 250.1327
====> Epoch: 11 Average loss: 190.4066
====> Epoch: 12 Average loss: 145.4134
====> Epoch: 13 Average loss: 111.6121
====> Epoch: 14 Average loss: 86.2275
====> Epoch: 15 Average loss: 66.6903
====> Epoch: 16 Average loss: 51.3354
====> Epoch: 17 Average loss: 39.9002
====> Epoch: 18 Average loss: 31.6693
====> Epoch: 19 Average loss: 25.5448
====> Epoch: 20 Average loss: 20.7542
====> Epoch: 21 Average loss: 16.8391
====> Epoch: 22 Average loss: 13.6953
====> Epoch: 23 Average loss: 11.2283
====> Epoch: 24 Average loss: 9.2945
====> Epoch: 25 Average loss: 7.7734
====

In [61]:
mse_loss = nn.MSELoss()

In [66]:
def loss_function(W, x, recons_x, h, la):
    mse = mse_loss(recons_x, x)
    # Since: W is shape of N_hidden x N. So, we do not need to transpose it as
    # opposed to #1
    dh = h * (1 - h) # Hadamard product produces size N_batch x N_hidden
    # Sum through the input dimension to improve efficiency, as suggested in #1
    w_sum = torch.sum(Variable(W)**2, dim=1)
    # unsqueeze to avoid issues with torch.mv
    w_sum = w_sum.unsqueeze(1) # shape N_hidden x 1
    contractive_loss = torch.sum(torch.mm(dh**2, w_sum), 0)
    return mse + contractive_loss.mul_(la)

In [3]:
noise = torch.randn(32, 100)
noise

tensor([[-1.2159, -0.3646, -0.9339,  ...,  0.3568,  0.5961,  0.9854],
        [-0.9196,  2.0576,  0.3647,  ...,  0.3570, -0.5031,  0.2708],
        [-0.8348, -0.8598,  0.5489,  ..., -0.9993,  0.2488, -1.7690],
        ...,
        [-0.3212, -1.2918,  1.1701,  ..., -0.9096,  0.9177, -0.8631],
        [-0.3914,  0.0542, -0.7175,  ..., -0.5089, -2.7883, -1.6865],
        [-0.2371, -0.8250,  1.5777,  ..., -0.8247, -1.2874, -0.6838]])

In [5]:
noise[0]

tensor([-1.2159, -0.3646, -0.9339,  1.0775, -1.3935, -0.3554, -1.5090, -1.1402,
         0.4547,  1.1131, -2.5994, -0.1088, -1.0914,  0.0175,  0.6703, -0.0524,
        -0.6850,  1.2449, -1.4801, -0.5576, -0.8470, -1.7579,  0.0273, -0.4125,
         0.9999, -1.5361, -0.9563, -0.1108,  0.8380, -0.4845, -0.0713,  0.1374,
         0.9887,  0.0235,  0.6651, -0.2038,  0.1207,  0.2328,  0.7428,  1.8720,
        -0.5450,  0.1125,  0.0743,  1.7451,  1.2407,  0.1858, -0.3367, -0.5417,
        -0.1951,  0.8974,  1.6717, -0.6741, -0.9113, -1.5281, -0.4455, -0.1984,
         1.6414,  0.4456, -0.8418, -0.6172,  0.6448, -2.0627, -2.0632,  1.3071,
        -0.3457,  0.7776,  0.3327, -0.8589,  0.5490, -0.7831, -0.6854,  1.5456,
        -0.8827, -1.3156, -0.3969, -0.3059,  1.2466,  1.1306, -0.5645, -0.2658,
         1.1909,  0.9103,  0.1147, -0.3577,  0.2128,  0.1694,  0.1826,  0.8907,
        -2.1180, -1.0883,  1.6153, -0.0655, -1.4373,  2.3685, -0.3375, -0.4674,
        -0.0172,  0.3568,  0.5961,  0.98

In [6]:
labels_onehot = torch.zeros(32, 10)
labels_onehot

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],


In [7]:
labels_onehot.scatter_(1, labels.view(batch_size, 1), 1)

NameError: name 'labels' is not defined